# 08. LLM 기반 부정 리뷰 세분류

## 목적
부정 리뷰(voted_up=0)를 7개 카테고리 + 톤/감성 메타데이터로 세분류한다.
"부정 리뷰가 많다"를 넘어 **"어떤 종류의 불만이, 얼마나, 어떤 톤으로"** 존재하는지 정량화.

## 왜 LLM 분류가 필요한가?
| 기존 분석 | 한계 |
|----------|------|
| TF-IDF(05편) | 긍정/부정 가르는 키워드는 알지만, 부정 내부 구성은 모름 |
| LDA(06편) | 토픽 추출했지만, 토픽 ≠ 불만 카테고리 (해석 모호) |
| 공기어(07편) | 단어 동시출현은 보이지만, 감정적 분풀이 vs 건설적 비판 구분 불가 |
| **LLM 분류** | **맥락을 읽고 카테고리 + 톤 + 실제감성까지 판단** |

## 분류 체계

### 카테고리 (7개)
| 카테고리 | 설명 | 예시 |
|---------|------|------|
| gameplay | 조작, 전투, 보스전, 미니게임, 밸런스 | boss fights are unfair |
| story_content | 스토리, 캐릭터, 엔딩, 세계관 | story gets boring after ch.3 |
| repetition | 루프 반복, 콘텐츠 부족, 엔드게임 | dive-cook-repeat |
| technical | 버그, 크래시, 최적화, 컨트롤러 | keeps crashing on startup |
| company | 퍼블리셔(넥슨), 가격, DLC 정책 | nexon will ruin this game |
| forced_neg | 비추천 테러, 장난, 역추천 | too good to recommend |
| other | 위에 해당 안 되는 것 | |

### 추가 메타데이터 (측면기반 감성분석 한계 보완)
| 필드 | 값 | 효과 |
|------|------|------|
| sub_category | 7개 중 하나 or none | 복합 불만 캡처 (보스전+반복 등) |
| tone | constructive / emotional / mixed / troll | 기획자용 필터 — 건설적 피드백만 추출 |
| actual_sentiment | pure_neg / mixed_neg / positive_but_neg_vote / joke | 긍정인데 비추 포착 |

## 분류 방법 (3종, DB에 method별 따로 저장)
| 방법 | 모델 | 비용 | 비고 |
|------|------|------|------|
| rules | 키워드 규칙 | $0 | baseline, 즉시 완료 |
| gemini | Gemini 2.5 Pro | Pro 요금제 | 구조화 JSON 출력 |
| anthropic | Claude Sonnet | ~$2.53 | Pydantic 구조화 출력 |

## 비용 추정 ($5 예산)
| 작업 | Sonnet 비용 | 누적 |
|------|-----------|------|
| 영어+한국어 부정 (1,947건) | ~$2.53 | $2.53 |
| 일본어 부정 (~200건) | ~$0.26 | $2.79 |
| 여유분 | | $2.21 남음 |

## DB 설계 원칙
- **기존 테이블(reviews, cleaned_reviews_en/ko) 절대 수정 안 함**
- `neg_reviews_classified` 새 테이블에 저장
- **review_id + classify_method 복합키** → 같은 리뷰를 3가지 방법으로 각각 저장
- 원본 reviews와 JOIN 가능 (review_id 기준)
- 배치마다 즉시 DB 저장 → 중간에 끊겨도 이어하기 가능

## 이 DB로 답할 수 있는 질문
1. 메인스토리 종료(20h 전후) 감성 분기
2. 라이트 유저(2-10h) 긍정률 원인
3. 개발자가 응답한/놓친 불만 카테고리
4. 2시간 미만 환불 미실행 유저 패턴
5. weighted_vote_score 높은(영향력 큰) 부정 리뷰 카테고리
6. 감정적 분풀이 vs 건설적 비판 비율

## 분석 순서
1. 라이브러리 & 데이터 로드
2. 부정 리뷰 필터링 & 메타데이터 조인
3. 기초 통계
4. 분류 체계 & 프롬프트 정의
5. 분류 함수 정의 (3종)
6. 분류 엔진 (배치 처리 + DB 즉시 저장)
7. 분류 실행
8. 분류 결과 분석 & 시각화
9. 플레이타임 구간별 불만 구성
10. 영어 vs 한국어 비교
11. 3모델 비교 분석
12. DB 연결 종료 & 추후 진행사항

---
## 1. 라이브러리 & 데이터 로드

In [1]:
import sys
import os
import platform
import sqlite3
import pandas as pd
import numpy as np
import json
import time
import re
import warnings
from collections import Counter
from tqdm import tqdm

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore')

if platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:
    plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

tqdm.pandas()

sys.path.append('..')
from config import DB_PATH, EVENTS, PLAYTIME_SEGMENTS

figures_dir = os.path.join('..', 'reports', 'figures')
os.makedirs(figures_dir, exist_ok=True)

db_path = os.path.join('..', DB_PATH)
conn = sqlite3.connect(db_path)

print(f'DB 연결 완료: {db_path}')

DB 연결 완료: ../data/dave_diver.db


---
## 2. 부정 리뷰 필터링 & 메타데이터 조인

cleaned_reviews + 원본 reviews JOIN → 풍부한 분석 테이블 생성.

| 메타 컬럼 | 용도 |
|----------|------|
| votes_up, weighted_vote_score | 영향력 큰 리뷰 식별 |
| developer_response | 개발자 응답 패턴 분석 |
| received_for_free, steam_purchase | 구매 유형별 분석 |
| playtime_hours | 20시간 전후 감성 분기 |
| language (원본) | 일본어 등 다국어 확장 |

In [2]:
def load_negative_reviews(conn, cleaned_table: str):
    """cleaned_reviews + reviews 원본 메타데이터 JOIN"""
    return pd.read_sql(f"""
        SELECT
            c.review_id,
            c.voted_up,
            c.review_month,
            c.play_segment,
            c.playtime_at_review,
            c.playtime_hours,
            c.received_for_free,
            c.written_during_early_access,
            c.review_text,
            c.cleaned_text,
            r.language,
            r.votes_up,
            r.votes_funny,
            r.weighted_vote_score,
            r.steam_purchase,
            r.developer_response,
            r.timestamp_dev_responded,
            r.num_games_owned,
            r.num_reviews
        FROM {cleaned_table} c
        LEFT JOIN reviews r ON c.review_id = r.review_id
        WHERE c.voted_up = 0
          AND c.cleaned_text IS NOT NULL
          AND c.cleaned_text != ''
    """, conn)


df_neg_en = load_negative_reviews(conn, 'cleaned_reviews_en')
df_neg_ko = load_negative_reviews(conn, 'cleaned_reviews_ko')

for df in [df_neg_en, df_neg_ko]:
    df['has_dev_response'] = df['developer_response'].notna().astype(int)
    df['text_len'] = df['review_text'].str.len()

print(f'영어 부정 리뷰:   {len(df_neg_en):,}건')
print(f'한국어 부정 리뷰: {len(df_neg_ko):,}건')
print(f'합계:             {len(df_neg_en) + len(df_neg_ko):,}건')

영어 부정 리뷰:   1,649건
한국어 부정 리뷰: 298건
합계:             1,947건


In [3]:
for name, df in [('영어', df_neg_en), ('한국어', df_neg_ko)]:
    total = len(df)
    over_500  = (df['text_len'] > 500).sum()
    over_1000 = (df['text_len'] > 1000).sum()
    over_2000 = (df['text_len'] > 2000).sum()
    
    print(f"\n[{name} 부정 리뷰] 총 {total:,}건")
    print(f"  500자 초과:  {over_500:,}건 ({over_500/total*100:.1f}%)")
    print(f"  1000자 초과: {over_1000:,}건 ({over_1000/total*100:.1f}%)")
    print(f"  2000자 초과: {over_2000:,}건 ({over_2000/total*100:.1f}%)")
    print(f"  평균 길이:   {df['text_len'].mean():.0f}자")
    print(f"  중앙값:      {df['text_len'].median():.0f}자")
    print(f"  최대:        {df['text_len'].max():,}자")


[영어 부정 리뷰] 총 1,649건
  500자 초과:  577건 (35.0%)
  1000자 초과: 278건 (16.9%)
  2000자 초과: 105건 (6.4%)
  평균 길이:   611자
  중앙값:      302자
  최대:        7,415자

[한국어 부정 리뷰] 총 298건
  500자 초과:  22건 (7.4%)
  1000자 초과: 7건 (2.3%)
  2000자 초과: 3건 (1.0%)
  평균 길이:   171자
  중앙값:      66자
  최대:        3,737자


- API 비용 문제로 리뷰 텍스트를 500자로 제한했는데, 분류 후 한국어는 전체, 영어는 샘플링으로 결과를 검증했다. 10개 중 1개 정도로 tone이 emotional임에도 constructive로 분류되는 오류가 있었지만, 감정적인 표현은 주로 리뷰 후반부에 집중되어 있었고 앞부분의 내용 자체는 실제로 건설적인 경우가 많았다. 건설적인 비판으로 시작해서 마지막에 넥슨 또는 특정 요소를 비판하는 패턴이 종종 관찰되었는데, 이는 오히려 500자 제한이 도움이 되었다. 

---
## 3. 기초 통계

In [10]:
segment_order = ['casual(<2h)', 'regular(2-10h)', 'engaged(10-30h)', 'engaged(30-50h)', 'hardcore(50h+)']

total_en = pd.read_sql("SELECT COUNT(*) as n FROM cleaned_reviews_en", conn).iloc[0, 0]
total_ko = pd.read_sql("SELECT COUNT(*) as n FROM cleaned_reviews_ko", conn).iloc[0, 0]

print('=== 부정 리뷰 현황 ===')
print(f'영어:   {len(df_neg_en):,} / {total_en:,} ({len(df_neg_en)/total_en*100:.1f}%)')
print(f'한국어: {len(df_neg_ko):,} / {total_ko:,} ({len(df_neg_ko)/total_ko*100:.1f}%)')

for label, neg_df in [('영어', df_neg_en), ('한국어', df_neg_ko)]:
    print(f'\n--- {label} 부정: 플레이타임 구간별 ---')
    for seg in segment_order:
        n = len(neg_df[neg_df['play_segment'] == seg])
        print(f'  {seg:10s}: {n:>5,}건')

dev_en = df_neg_en['has_dev_response'].sum()
dev_ko = df_neg_ko['has_dev_response'].sum()
print(f'\n--- 개발자 응답 있는 부정 리뷰 ---')
print(f'영어: {dev_en}건 / {len(df_neg_en):,}건 ({dev_en/len(df_neg_en)*100:.1f}%)')
print(f'한국어: {dev_ko}건 / {len(df_neg_ko):,}건 ({dev_ko/max(len(df_neg_ko),1)*100:.1f}%)')

print(f'\n--- 텍스트 길이 ---')
print(f'영어:   중앙값 {df_neg_en["text_len"].median():.0f}자, 20자 미만 {(df_neg_en["text_len"]<20).sum()}건')
print(f'한국어: 중앙값 {df_neg_ko["text_len"].median():.0f}자, 10자 미만 {(df_neg_ko["text_len"]<10).sum()}건')

=== 부정 리뷰 현황 ===
영어:   1,649 / 44,839 (3.7%)
한국어: 298 / 10,074 (3.0%)

--- 영어 부정: 플레이타임 구간별 ---
  casual(<2h):   367건
  regular(2-10h):   390건
  engaged(10-30h):   564건
  engaged(30-50h):   188건
  hardcore(50h+):   140건

--- 한국어 부정: 플레이타임 구간별 ---
  casual(<2h):    38건
  regular(2-10h):    63건
  engaged(10-30h):   133건
  engaged(30-50h):    38건
  hardcore(50h+):    26건

--- 개발자 응답 있는 부정 리뷰 ---
영어: 285건 / 1,649건 (17.3%)
한국어: 66건 / 298건 (22.1%)

--- 텍스트 길이 ---
영어:   중앙값 302자, 20자 미만 69건
한국어: 중앙값 66자, 10자 미만 12건


---
## 4. 분류 체계 & 프롬프트 정의

모든 LLM이 공유하는 카테고리, 프롬프트, 파서.

In [11]:
# ══════════════════════════════════════════════════════════════
#  분류 체계 정의
# ══════════════════════════════════════════════════════════════

#리뷰를 분류할 7가지 주요 카테고리 목록을 정의
# 게임플레이, 스토리/컨텐츠, 반복(루프), 기술적문제, 회사(넥슨), 스토리강제, 그외
CATEGORIES = [
    'gameplay', 'story_content', 'repetition',
    'technical', 'company', 'forced_neg', 'other',
]

VALID_CATEGORIES = set(CATEGORIES)#카테고리 목록을 집합으로 변환

#리뷰의 어조를 분류하기 위한 4가지 기준
# 건설적인, 감정적인, 복합적, 트롤(단순 반복, 도배 등)
VALID_TONES = {'constructive', 'emotional', 'mixed', 'troll'}

#실제 감성(sentiment)를 분류하기 위한 4가지 기준
#강한부정, 복합적 부정, 긍정적인 어조 하지만 부정 투표, 농담식 부정
VALID_SENTIMENTS = {'pure_negative', 'mixed_negative', 'positive_but_negative_vote', 'joke'}


def build_classification_prompt(reviews_batch: list[dict], language: str) -> str:
    """
    분류 프롬프트 생성.
    
    v2 설계:
    - category + sub_category: 복합 불만 캡처
    - tone: constructive vs emotional(건설적인 vs 감정적인) -> 기획자용 필터
    - actual_sentiment: 긍정인데 비추천 포착
    - 한국어 은어 가이드 포함
    - 모호한 경우 판단 기준 명시 → other 남발 방지
    """
    
    #요청된 언어가 한국어일 경우 한국어 특유의 맥락과 은어 주의사항을 추가하고, 아니면 영어 안내문을 쓴다.
    lang_note = ('리뷰는 한국어입니다. 한국어 맥락을 고려하세요. '
                 '한국어 인터넷 은어(갓겜, 망겜, 현질 등)에 주의하세요.'
                 if language == 'korean'
                 else 'Reviews are in English.')

    # 들어온 리뷰들을 프롬포트에 넣기 위해 문자열로 결합할 변수 초기화
    reviews_text = ''
    for item in reviews_batch:#리뷰 리스트 순회
        text = item['text'][:500].replace('"', "'")#각 리뷰의 텍스트를 500자 까지 짜르기
        reviews_text += f'[{item["idx"]}] {text}\n\n'


# LLM에 전달할 전체 프롬포트 문자열을 구성
    prompt = f"""You are an expert game review analyst classifying negative Steam reviews of "Dave the Diver".
{lang_note}

## TASK
For each review, determine:
1. **category**: The PRIMARY complaint (1 of 7)
2. **sub_category**: Secondary complaint if present (same 7, or "none")
3. **tone**: Is this constructive feedback or emotional venting?
4. **actual_sentiment**: Does the reviewer actually hate the game, or is it more nuanced?
5. **confidence** and **reason**

## Categories (choose exactly ONE for category)

gameplay: Controls, combat, boss fights, minigames, difficulty, balance, weapon/upgrade mechanics.
  - Includes: unfair bosses, clunky controls, forced minigames, bad balance, farming tedium.
  - Does NOT include: general "boring" (-> repetition) or story complaints.

story_content: Story, characters, ending, world-building, dialogue, pacing, quests/missions.
  - Includes: weak ending, boring story after chapter X, flat characters.
  - Does NOT include: lack of endgame content (-> repetition).

repetition: Repetitive game loop, lack of content, no endgame, gets boring over time.
  - Includes: dive-cook-repeat, same routine, nothing new after X hours, tedious grind.
  - Key signal: complaints about SAMENESS or CONTENT RUNNING OUT.

technical: Bugs, crashes, loading, optimization, controller issues, save corruption.
  - Includes: Steam Deck/Linux issues, performance drops, won't launch.

company: Publisher (Nexon), pricing, DLC policy, monetization fears, corporate decisions.
  - Includes: "Nexon will ruin it", overpriced, anti-consumer.

forced_neg: NOT a genuine complaint. Protest vote, joke, trolling, reverse psychology.
  - Key signal: the review text is actually positive or absurd, but voted negative.

other: Genuinely negative but doesn't fit above. Personal taste, vague, unrelated.
  - Use sparingly. Prefer a specific category when possible.

## Tone classification
- constructive: Specific criticism with identifiable issues. Useful for developers.
- emotional: Venting, anger, frustration without specific actionable feedback.
- mixed: Has both specific points AND emotional language.
- troll: Joke review, meme, copypasta, or intentionally absurd.

## Actual sentiment (catches mismatches)
- pure_negative: Genuinely dislikes the game. The negative vote matches the content.
- mixed_negative: Acknowledges good parts but overall negative. "Fun at first but..."
- positive_but_negative_vote: Review TEXT is positive/praising but voted negative.
- joke: Not a real review. Meme, copypasta, single emoji, etc.

## Decision rules
- "Boring" alone -> repetition (unless specifically about story -> story_content)
- "Not worth the price" -> company
- Multiple complaints -> category = PRIMARY, sub_category = SECONDARY
- "Game is great BUT Nexon..." -> category=company, actual_sentiment=positive_but_negative_vote
- "Best game ever [thumbs down]" -> category=forced_neg, actual_sentiment=positive_but_negative_vote
- Short emotional rant with no specifics -> tone=emotional, category=best guess or other
- Korean slang: 갓겜=great game, 망겜=ruined game, 현질=microtransaction, 노가다=grinding

## Reviews to classify

{reviews_text}

## Output format
Return ONLY a JSON array. Each element:
- "idx": review number
- "category": one of [gameplay, story_content, repetition, technical, company, forced_neg, other]
- "sub_category": same options OR "none"
- "tone": one of [constructive, emotional, mixed, troll]
- "actual_sentiment": one of [pure_negative, mixed_negative, positive_but_negative_vote, joke]
- "confidence": high / medium / low
- "reason": 1-sentence explanation in English

Return {len(reviews_batch)} items. JSON array only, no markdown fences."""

    return prompt


# ── JSON 스키마 (Gemini 구조화 출력용) ──
REVIEW_JSON_SCHEMA = {
    'type': 'ARRAY',
    'items': {
        'type': 'OBJECT',
        'required': ['idx', 'category', 'sub_category', 'tone', 'actual_sentiment', 'confidence', 'reason'],
        'properties': {
            'idx': {'type': 'INTEGER'},
            'category': {'type': 'STRING', 'enum': CATEGORIES},
            'sub_category': {'type': 'STRING', 'enum': CATEGORIES + ['none']},
            'tone': {'type': 'STRING', 'enum': list(VALID_TONES)},
            'actual_sentiment': {'type': 'STRING', 'enum': list(VALID_SENTIMENTS)},
            'confidence': {'type': 'STRING', 'enum': ['high', 'medium', 'low']},
            'reason': {'type': 'STRING'},
        }
    }
}


def parse_llm_response(response_text: str, expected_count: int) -> list[dict]:
    """
    LLM 응답 파싱. 3단계 fallback.
    Gemini 구조화 출력 사용 시 1차에서 거의 항상 성공.
    """
    default_item = {
        'category': 'other', 'sub_category': 'none',
        'tone': 'mixed', 'actual_sentiment': 'pure_negative',
        'confidence': 'low', 'reason': 'parse_failed',
    }

# 개별 파싱된 아이템의 값이 유효한지 검증하고 정제하는 내부 함수입니다.
    def validate_item(item: dict) -> dict:
        # 딕셔너리에서 값을 가져오되, 없으면 기본값을 할당합니다.
        cat = item.get('category', 'other')
        sub = item.get('sub_category', 'none')
        tone = item.get('tone', 'mixed')
        sentiment = item.get('actual_sentiment', 'pure_negative')
        
        # 가져온 값이 사전에 정의한 집합(VALID_*)에 포함되는지 확인하고, 유효한 딕셔너리로 묶어 반환합니다.
        return {
            'category': cat if cat in VALID_CATEGORIES else 'other',
            'sub_category': sub if (sub in VALID_CATEGORIES or sub == 'none') else 'none',
            'tone': tone if tone in VALID_TONES else 'mixed',
            'actual_sentiment': sentiment if sentiment in VALID_SENTIMENTS else 'pure_negative',
            'confidence': item.get('confidence', 'unknown'),
            'reason': str(item.get('reason', ''))[:200], # 이유는 최대 200자까지만 자릅니다.
        }

# ── 1차: 전체 JSON 파싱 ──
    # 응답 텍스트의 앞뒤 공백을 제거합니다.
    cleaned = response_text.strip()
    # 마크다운 코드 블록 시작 기호(```json 또는 ```)를 정규식으로 제거합니다.
    cleaned = re.sub(r'^```(?:json)?\s*', '', cleaned)
    # 마크다운 코드 블록 종료 기호(```)를 제거합니다.
    cleaned = re.sub(r'\s*```$', '', cleaned)


    try:
        results = json.loads(cleaned)
        if isinstance(results, list):
            validated = [validate_item(item) for item in results]
            while len(validated) < expected_count:
                validated.append({**default_item, 'reason': 'parse_error: missing'})
            return validated[:expected_count]
    except json.JSONDecodeError:
        pass

    # ── 2차: JSON 배열 부분 추출 ──
    json_match = re.search(r'\[.*\]', cleaned, re.DOTALL)
    if json_match:
        try:
            results = json.loads(json_match.group())
            if isinstance(results, list):
                validated = [validate_item(item) for item in results]
                while len(validated) < expected_count:
                    validated.append({**default_item, 'reason': 'partial_json'})
                return validated[:expected_count]
        except json.JSONDecodeError:
            pass

    # ── 3차: 정규식으로 category만이라도 추출 ──
    cats = re.findall(r'"category"\s*:\s*"(\w+)"', response_text)
    tones = re.findall(r'"tone"\s*:\s*"(\w+)"', response_text)
    sents = re.findall(r'"actual_sentiment"\s*:\s*"([\w]+)"', response_text)

    if cats:
        results = []
        for i, cat in enumerate(cats):
            results.append({
                'category': cat if cat in VALID_CATEGORIES else 'other',
                'sub_category': 'none',
                'tone': tones[i] if i < len(tones) and tones[i] in VALID_TONES else 'mixed',
                'actual_sentiment': sents[i] if i < len(sents) and sents[i] in VALID_SENTIMENTS else 'pure_negative',
                'confidence': 'low',
                'reason': 'regex_parse',
            })
        while len(results) < expected_count:
            results.append({**default_item, 'reason': 'regex_incomplete'})
        return results[:expected_count]

    return [dict(default_item) for _ in range(expected_count)]


print(f'카테고리 {len(CATEGORIES)}개, 톤 {len(VALID_TONES)}개, 감성 {len(VALID_SENTIMENTS)}개')
print(f'Gemini JSON 스키마 정의 완료')

카테고리 7개, 톤 4개, 감성 4개
Gemini JSON 스키마 정의 완료


## 프롬포트 전문 한국어 정리 

- 당신은 "데이브 더 다이버(Dave the Diver)"의 부정적인 스팀 리뷰를 분류하는 게임 리뷰 분석 전문가입니다.
(언어 조건에 따라: 리뷰는 한국어입니다. 한국어 맥락을 고려하세요. 한국어 인터넷 은어(갓겜, 망겜, 현질 등)에 주의하세요. OR 리뷰는 영어입니다.)

## 작업 (TASK)
각 리뷰에 대해 다음 사항을 결정하세요:
1. **category (카테고리)**: 가장 주된 불만 사항 (7가지 중 1개)
2. **sub_category (서브 카테고리)**: 부차적인 불만 사항이 있을 경우 (동일한 7가지 중 1개, 없으면 "none")
3. **tone (어조)**: 건설적인 피드백인가, 아니면 감정적인 분출인가?
4. **actual_sentiment (실제 감성)**: 리뷰어가 실제로 게임을 싫어하는가, 아니면 복합적인 감정인가?
5. **confidence (신뢰도)** 및 **reason (이유)**

## 카테고리 (category에는 정확히 1개만 선택)

* **gameplay (게임플레이)**: 조작감, 전투, 보스전, 미니게임, 난이도, 밸런스, 무기/업그레이드 시스템.
    * 포함: 불합리한 보스, 뻣뻣한 조작감, 강제되는 미니게임, 엉망인 밸런스, 지루한 파밍.
    * 제외: 단순한 "지루함" (-> repetition으로 분류) 또는 스토리 관련 불만.
* **story_content (스토리/콘텐츠)**: 스토리, 캐릭터, 엔딩, 세계관, 대화, 전개 속도, 퀘스트/미션.
    * 포함: 빈약한 엔딩, 챕터 X 이후로 지루해진 스토리, 평면적인 캐릭터.
    * 제외: 엔드 콘텐츠 부족 (-> repetition으로 분류).
* **repetition (반복성)**: 반복되는 게임 루프, 콘텐츠 부족, 엔드 콘텐츠 부재, 갈수록 지루해짐.
    * 포함: 잠수-요리-반복, 똑같은 루틴, X시간 이후 새로운 게 없음, 지루한 노가다.
    * 핵심 신호: '똑같음(SAMENESS)' 또는 '콘텐츠가 바닥남'에 대한 불만.
* **technical (기술적 문제)**: 버그, 크래시(튕김), 로딩, 최적화, 컨트롤러 인식 문제, 세이브 파일 손상.
    * 포함: 스팀덱/리눅스 호환 문제, 프레임 드랍, 실행 불가.
* **company (개발사/퍼블리셔)**: 퍼블리셔(넥슨), 가격 정책, DLC 정책, 과금 유도에 대한 우려, 기업의 결정.
    * 포함: "넥슨이 망칠 거다", 너무 비싸다, 소비자 기만.
* **forced_neg (거짓 부정)**: 진짜 불만이 아님. 항의성 투표, 장난, 트롤링, 반어법(역설적 칭찬).
    * 핵심 신호: 리뷰 텍스트는 긍정적이거나 터무니없는데, '비추천'을 누른 경우.
* **other (기타)**: 진심으로 부정적인 리뷰지만 위 항목에 맞지 않음. 개인적 취향, 모호함, 게임과 무관함.
    * 최소한으로만 사용할 것. 가급적 구체적인 카테고리를 우선 선택할 것.

## 어조 분류 (Tone)
* **constructive (건설적)**: 문제점이 명확한 구체적인 비판. 개발자에게 유용한 피드백.
* **emotional (감정적)**: 구체적이고 실천 가능한 피드백이 없는 단순 화풀이, 분노, 좌절.
* **mixed (혼합)**: 구체적인 지적과 감정적인 언어가 섞여 있음.
* **troll (트롤)**: 장난성 리뷰, 밈(Meme), 복붙(Copypasta), 또는 의도적으로 터무니없는 내용.

## 실제 감성 (Actual sentiment - 불일치 잡아내기)
* **pure_negative (순수 부정)**: 진심으로 게임을 싫어함. 비추천 투표와 내용이 일치함.
* **mixed_negative (혼합 부정)**: 좋은 점도 인정하지만 전반적으로 부정적임. "처음엔 재밌었지만..."
* **positive_but_negative_vote (긍정 내용+비추천 투표)**: 리뷰 '내용'은 긍정적이고 칭찬하지만 '비추천'을 누른 경우.
* **joke (농담)**: 제대로 된 리뷰가 아님. 밈, 복붙 내용, 이모티콘 한 개 등.

## 판단 규칙 (Decision rules)
* 단순히 "지루하다" -> repetition (스토리를 구체적으로 언급하지 않는 한)
* "돈값 못한다" -> company
* 불만이 여러 개일 경우 -> category = 가장 주된 불만, sub_category = 부차적 불만
* "게임은 훌륭한데 넥슨이..." -> category=company, actual_sentiment=positive_but_negative_vote
* "최고의 게임 [비추천]" -> category=forced_neg, actual_sentiment=positive_but_negative_vote
* 구체적인 내용 없이 짧게 감정만 쏟아낸 경우 -> tone=emotional, category=가장 그럴싸한 것 또는 other
* 한국어 은어 처리: 갓겜=great game, 망겜=ruined game, 현질=microtransaction, 노가다=grinding

## 분류할 리뷰 데이터
[입력된 리뷰 데이터 목록 삽입됨]

## 출력 형식 (Output format)
오직 JSON 배열만 반환하세요. 각 요소의 형태:
- "idx": 리뷰 번호
- "category": [gameplay, story_content, repetition, technical, company, forced_neg, other] 중 하나
- "sub_category": 위와 동일 또는 "none"
- "tone": [constructive, emotional, mixed, troll] 중 하나
- "actual_sentiment": [pure_negative, mixed_negative, positive_but_negative_vote, joke] 중 하나
- "confidence": high / medium / low
- "reason": 영어로 작성된 1문장 분량의 이유

총 {배치 크기}개의 항목을 반환하세요. 마크다운 기호 없이 JSON 배열만 출력하세요.

---
## 5. 분류 함수 정의 (3종)

### 5-1. Anthropic API (Claude Sonnet)

```bash
pip install anthropic
export ANTHROPIC_API_KEY='your-key'
```

In [17]:
def classify_batch_anthropic(reviews_batch: list[dict], language: str,
                              model: str = 'claude-sonnet-4-20250514') -> list[dict]:
    """
    Anthropic API 배치 분류.
    Sonnet: $3/MTok input, $15/MTok output
    """
    import anthropic
    client = anthropic.Anthropic()  # ANTHROPIC_API_KEY 환경변수에서 자동 로드

    prompt = build_classification_prompt(reviews_batch, language)

    response = client.messages.create(
        model=model,
        max_tokens=4096,
        messages=[{'role': 'user', 'content': prompt}]
    )

    return parse_llm_response(response.content[0].text, len(reviews_batch))


print('Anthropic (Sonnet) 분류 함수 정의 완료')

Anthropic (Sonnet) 분류 함수 정의 완료


### 5-2. Google Gemini API (Pro)

```bash
pip install google-genai
export GEMINI_API_KEY='your-key'
```

**Gemini의 핵심 장점: `response_json_schema`로 구조화된 JSON 출력을 강제.**
LLM이 스키마에 맞지 않는 응답을 생성할 수 없으므로 파싱 실패가 거의 없다.

In [18]:
def classify_batch_gemini(reviews_batch: list[dict], language: str,
                           model: str = 'gemini-2.5-flash') -> list[dict]:
    """
    Google Gemini API 배치 분류.
    response_json_schema로 구조화 출력 → 파싱 실패 최소화.
    """
    from google import genai
    from google.genai import types

    api_key = os.environ.get('GEMINI_API_KEY')
    if not api_key:
        raise ValueError('GEMINI_API_KEY 환경변수가 설정되지 않았습니다.')

    client = genai.Client(api_key=api_key)

    prompt = build_classification_prompt(reviews_batch, language)

    response = client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.1,
            max_output_tokens=4096,
            response_mime_type='application/json',
            response_json_schema=REVIEW_JSON_SCHEMA,
        )
    )

    return parse_llm_response(response.text, len(reviews_batch))


print('Gemini (무료) 분류 함수 정의 완료')

Gemini (무료) 분류 함수 정의 완료


### 5-3. 키워드 규칙 기반 (Fallback)

LLM 없이 동작. baseline 역할.
tone/actual_sentiment는 규칙으로 제한적으로만 판단.

In [15]:
# ══════════════════════════════════════════════════════════════
#  키워드 및 규칙 정의
# ══════════════════════════════════════════════════════════════

# 카테고리별로 영어(en)와 한국어(ko) 매칭 키워드를 정의한 딕셔너리입니다.
KEYWORD_RULES = {
    'forced_neg': { ... }, # (생략: 억까, 비추테러 관련 키워드)
    'technical': { ... },  # (생략: 버그, 튕김 등 기술적 문제 키워드)
    'company': { ... },    # (생략: 넥슨, 돈, 퍼블리셔 등 기업 관련 키워드)
    'repetition': { ... }, # (생략: 반복, 지루, 콘텐츠 부족 등 키워드)
    'gameplay': { ... },   # (생략: 보스, 전투, 조작, 난이도 등 키워드)
    'story_content': { ... }, # (생략: 스토리, 엔딩, 캐릭터 등 키워드)
}

# 키워드가 여러 카테고리에서 겹칠 경우, 먼저 매칭할 우선순위를 정의합니다.
RULE_PRIORITY = ['forced_neg', 'technical', 'company', 'repetition', 'gameplay', 'story_content']

# ── 감성 판단용 긍정 키워드 (forced_neg / positive_but_negative_vote 감지) ──
# 비추천 리뷰인데 글 내용에 긍정적인 단어가 있는지 확인하기 위한 키워드입니다.
POSITIVE_SIGNALS = {
    'en': ['love', 'great', 'amazing', 'wonderful', 'best game', 'masterpiece',
           'fun', 'enjoy', 'recommend', 'worth', '10/10', '9/10', 'fantastic'],
    'ko': ['갓겜', '명작', '재밌', '재미있', '추천', '최고', '좋아', '사랑',
           '꿀잼', '인생겜'],
}


# (수정) 원본 리뷰의 식별을 위해 idx를 인자로 추가로 받습니다.
def classify_with_rules(review_text: str, idx: int, language: str = 'english') -> dict:
    """키워드 규칙 기반 분류 (1건)"""
    # 언어 설정에 따라 딕셔너리 키(en 또는 ko)를 결정합니다.
    lang_key = 'ko' if language == 'korean' else 'en'
    # 대소문자 구분을 없애기 위해 리뷰 텍스트를 모두 소문자로 변환합니다.
    text_lower = review_text.lower()

    # ── 1. 주 카테고리(category) 분류 ──
    category = 'other' # 기본값은 other로 설정합니다.
    matched_kw = []    # 매칭된 키워드를 저장할 리스트입니다.
    # 우선순위가 높은 카테고리부터 순서대로 검사합니다.
    for cat in RULE_PRIORITY:
        keywords = KEYWORD_RULES[cat][lang_key]
        # 리뷰 텍스트 안에 포함된 키워드들을 모두 찾아 리스트로 만듭니다.
        hits = [kw for kw in keywords if kw in text_lower]
        if hits: # 매칭된 키워드가 하나라도 있다면
            category = cat
            matched_kw = hits[:3] # 로그를 남기기 위해 찾은 키워드를 최대 3개까지만 저장합니다.
            break # 우선순위가 높은 것을 찾았으므로 루프를 즉시 종료합니다.

    # ── 2. 서브 카테고리(sub_category) 분류 ──
    sub_category = 'none' # 기본값은 none으로 설정합니다.
    for cat in RULE_PRIORITY:
        if cat == category:
            continue # 이미 주 카테고리로 선정된 것은 건너뜁니다.
        keywords = KEYWORD_RULES[cat][lang_key]
        # 다른 카테고리의 키워드가 리뷰에 존재하는지 확인합니다.
        if any(kw in text_lower for kw in keywords):
            sub_category = cat
            break # 가장 먼저 찾은 부차적 불만을 서브 카테고리로 지정하고 종료합니다.

    # ── 3. 어조(tone) 분류 ──
    tone = 'mixed'  # 기본값
    if category == 'forced_neg':
        tone = 'troll' # 거짓 부정(억까/장난)인 경우 트롤로 간주합니다.
    elif len(text_lower) < 30:
        tone = 'emotional'  # (주의: 너무 짧은 리뷰를 감정적으로 단정짓는 논리적 허점이 있는 부분)

    # ── 4. 실제 감성(actual_sentiment) 분류 ──
    pos_signals = POSITIVE_SIGNALS[lang_key]
    # 리뷰 텍스트에 긍정 키워드가 하나라도 포함되어 있는지 검사합니다.
    has_positive = any(kw in text_lower for kw in pos_signals)
    
    # 카테고리가 억까(forced_neg)인데 긍정 키워드도 있다면 -> 사실상 긍정인데 비추 누른 것
    if category == 'forced_neg' and has_positive:
        actual_sentiment = 'positive_but_negative_vote'
    # 카테고리가 억까인데 긍정 키워드가 없다면 -> 그냥 농담/장난글
    elif category == 'forced_neg':
        actual_sentiment = 'joke'
    # 일반적인 불만인데 긍정 키워드가 섞여 있다면 -> 아쉬운 부정 (예: "재밌는데 버그가..")
    elif has_positive:
        actual_sentiment = 'mixed_negative'
    # 일반적인 불만이고 긍정 키워드도 없다면 -> 순수 부정
    else:
        actual_sentiment = 'pure_negative'

    # 분석 결과를 딕셔너리 형태로 반환합니다. (누락되었던 idx 추가)
    return {
        'idx': idx,
        'category': category,
        'sub_category': sub_category,
        'tone': tone,
        'actual_sentiment': actual_sentiment,
        'confidence': 'rule_based', # LLM이 아닌 규칙 기반임을 명시합니다.
        'reason': f'keyword_match: {", ".join(matched_kw)}' if matched_kw else 'no_keyword_match',
    }


def classify_batch_rules(reviews_batch: list[dict], language: str) -> list[dict]:
    """규칙 기반 배치 분류"""
    # 배치로 들어온 각 리뷰 딕셔너리에서 text와 idx를 뽑아내어 처리 후 리스트로 반환합니다.
    return [classify_with_rules(item['text'], item['idx'], language) for item in reviews_batch]


print('키워드 규칙 분류 함수 정의 완료')

키워드 규칙 분류 함수 정의 완료


---
## 6. 분류 엔진 (배치 처리 + DB 즉시 저장)

### 핵심 설계
- **review_id + classify_method 복합키** → 같은 리뷰를 3가지 방법으로 각각 저장
- 배치마다 DB에 즉시 저장 → 중간에 끊겨도 이어하기
- 이미 분류된 리뷰는 건너뜀
- LLM 에러 시 규칙 기반 fallback → 데이터 손실 없음
- **기존 테이블(reviews, cleaned_reviews_en/ko)은 절대 수정 안 함**

In [7]:
DB_TABLE = 'neg_reviews_classified'


def init_classification_table(conn):
    """분류 결과 테이블 생성 (없으면). 기존 테이블은 건드리지 않음."""
    conn.execute(f"""
        CREATE TABLE IF NOT EXISTS {DB_TABLE} (
            review_id           TEXT,
            classify_method     TEXT,
            language_group      TEXT,
            -- 원본 메타데이터 (reviews JOIN)
            voted_up            INTEGER,
            review_month        TEXT,
            play_segment        TEXT,
            playtime_at_review  INTEGER,
            playtime_hours      REAL,
            received_for_free   INTEGER,
            written_during_early_access INTEGER,
            language            TEXT,
            votes_up            INTEGER,
            votes_funny         INTEGER,
            weighted_vote_score REAL,
            steam_purchase      INTEGER,
            has_dev_response    INTEGER,
            num_games_owned     INTEGER,
            num_reviews         INTEGER,
            text_len            INTEGER,
            -- 텍스트
            review_text         TEXT,
            cleaned_text        TEXT,
            -- 분류 결과
            category            TEXT,
            sub_category        TEXT,
            tone                TEXT,
            actual_sentiment    TEXT,
            confidence          TEXT,
            reason              TEXT,
            PRIMARY KEY (review_id, classify_method)
        )
    """)
    conn.commit()


def get_already_classified(conn, method: str) -> set:
    """특정 method로 이미 분류된 review_id 반환"""
    try:
        result = pd.read_sql(
            f"SELECT review_id FROM {DB_TABLE} WHERE classify_method LIKE ?",
            conn, params=(f'{method}%',)
        )
        return set(result['review_id'].tolist())
    except Exception:
        return set()


def save_batch_to_db(conn, df_batch: pd.DataFrame, results: list[dict],
                      method: str, language_group: str):
    """배치 분류 결과를 즉시 DB에 저장"""
    rows = []
    for i, (_, row) in enumerate(df_batch.iterrows()):
        r = results[i] if i < len(results) else {
            'category': 'other', 'sub_category': 'none',
            'tone': 'mixed', 'actual_sentiment': 'pure_negative',
            'confidence': 'low', 'reason': 'index_error',
        }
        rows.append({
            'review_id': row['review_id'],
            'classify_method': method,
            'language_group': language_group,
            'voted_up': row['voted_up'],
            'review_month': row['review_month'],
            'play_segment': row['play_segment'],
            'playtime_at_review': row.get('playtime_at_review'),
            'playtime_hours': row.get('playtime_hours'),
            'received_for_free': row.get('received_for_free'),
            'written_during_early_access': row.get('written_during_early_access'),
            'language': row.get('language'),
            'votes_up': row.get('votes_up'),
            'votes_funny': row.get('votes_funny'),
            'weighted_vote_score': row.get('weighted_vote_score'),
            'steam_purchase': row.get('steam_purchase'),
            'has_dev_response': row.get('has_dev_response'),
            'num_games_owned': row.get('num_games_owned'),
            'num_reviews': row.get('num_reviews'),
            'text_len': row.get('text_len'),
            'review_text': row['review_text'],
            'cleaned_text': row['cleaned_text'],
            'category': r['category'],
            'sub_category': r.get('sub_category', 'none'),
            'tone': r.get('tone', 'mixed'),
            'actual_sentiment': r.get('actual_sentiment', 'pure_negative'),
            'confidence': r['confidence'],
            'reason': r['reason'],
        })

    pd.DataFrame(rows).to_sql(DB_TABLE, conn, if_exists='append', index=False)
    conn.commit()


def run_classification(conn, df_neg: pd.DataFrame, language_group: str,
                        method: str, batch_size: int = 25):
    """
    메인 분류 루프.
    - 이미 분류된 리뷰 건너뜀
    - 배치마다 DB 즉시 저장
    - LLM 에러 시 규칙 fallback
    """
    already = get_already_classified(conn, method)
    remaining = df_neg[~df_neg['review_id'].isin(already)].copy()

    if len(remaining) == 0:
        print(f'  [{language_group}] 이미 전부 분류 완료 ({len(df_neg):,}건). 건너뜀.')
        return

    skipped = len(already & set(df_neg['review_id']))
    print(f'  [{language_group}] 분류 대상: {len(remaining):,}건 (이미 완료: {skipped:,}건)')

    # 분류 함수 & sleep 설정
    if method == 'anthropic':
        classify_fn = lambda b, l: classify_batch_anthropic(b, l)
        sleep_time = 1.0
    elif method == 'gemini':
        classify_fn = lambda b, l: classify_batch_gemini(b, l)
        sleep_time = 4.5  # 무료 요금제
    else:  # rules
        classify_fn = classify_batch_rules
        sleep_time = 0

    total_batches = (len(remaining) + batch_size - 1) // batch_size

    for start in tqdm(range(0, len(remaining), batch_size),
                      total=total_batches, desc=f'{language_group} ({method})'):
        batch_df = remaining.iloc[start:start+batch_size]
        batch_input = [{'idx': j+1, 'text': row['review_text']}
                       for j, (_, row) in enumerate(batch_df.iterrows())]

        try:
            results = classify_fn(batch_input, language_group)
            save_batch_to_db(conn, batch_df, results, method, language_group)
        except Exception as e:
            print(f'\n  [에러] batch {start//batch_size}: {e}')
            fallback = classify_batch_rules(batch_input, language_group)
            save_batch_to_db(conn, batch_df, fallback, f'{method}_fallback', language_group)

        if sleep_time > 0:
            time.sleep(sleep_time)


print('분류 엔진 정의 완료')

분류 엔진 정의 완료


---
## 7. 분류 실행

In [ ]:
CLASSIFY_METHOD = 'anthropic'  # <- 여기를 변경
BATCH_SIZE = 20

print(f'분류 방법: {CLASSIFY_METHOD}')
print(f'배치 크기: {BATCH_SIZE}')

분류 방법: anthropic
배치 크기: 20


In [31]:
# ══════════════════════════════════════════════════════════════
# 분류 실행
# ══════════════════════════════════════════════════════════════

init_classification_table(conn)

print(f'=== 분류 시작 (method={CLASSIFY_METHOD}) ===')
print(f'※ 기존 reviews, cleaned_reviews 테이블은 수정하지 않습니다.')
print(f'※ 이미 같은 method로 분류된 리뷰는 건너뜁니다.')
print()

run_classification(conn, df_neg_en, 'english', CLASSIFY_METHOD, BATCH_SIZE)
run_classification(conn, df_neg_ko, 'korean', CLASSIFY_METHOD, BATCH_SIZE)

check = pd.read_sql(f"""
    SELECT classify_method, language_group, COUNT(*) as n
    FROM {DB_TABLE}
    GROUP BY classify_method, language_group
    ORDER BY classify_method, language_group
""", conn)
print(f'\n=== DB 현황 ({DB_TABLE}) ===')
print(check.to_string(index=False))

=== 분류 시작 (method=anthropic) ===
※ 기존 reviews, cleaned_reviews 테이블은 수정하지 않습니다.
※ 이미 같은 method로 분류된 리뷰는 건너뜁니다.

  [english] 분류 대상: 1,649건 (이미 완료: 0건)


english (anthropic): 100%|██████████| 83/83 [32:50<00:00, 23.74s/it]


  [korean] 분류 대상: 298건 (이미 완료: 0건)


korean (anthropic): 100%|██████████| 15/15 [06:17<00:00, 25.19s/it]


=== DB 현황 (neg_reviews_classified) ===
classify_method language_group    n
      anthropic        english 1649
      anthropic         korean  298
         gemini        english  340
          rules        english 1649
          rules         korean  298


In [32]:
# fallback만 삭제 (에러 나서 규칙으로 대체된 것만)
conn.execute("DELETE FROM neg_reviews_classified WHERE classify_method = 'gemini_fallback'")
conn.commit()

# 확인: gemini로 정상 저장된 건수
check = pd.read_sql("""
    SELECT classify_method, COUNT(*) as n 
    FROM neg_reviews_classified 
    GROUP BY classify_method
""", conn)
print(check.to_string(index=False))

classify_method    n
      anthropic 1947
         gemini  340
          rules 1947


---
## 8. 분류 결과 분석 & 시각화

DB에서 읽어서 분석. LLM을 다시 돌릴 필요 없음.

**분석 기준 method 선택:** 아래 셀에서 `ANALYSIS_METHOD`를 지정.
3모델 비교는 섹션 11에서.

In [16]:
# 분석에 사용할 method 선택
ANALYSIS_METHOD = 'anthropic'  # 또는 'gemini', 'anthropic'

classified = pd.read_sql(
    f"SELECT * FROM {DB_TABLE} WHERE classify_method = ?",
    conn, params=(ANALYSIS_METHOD,)
)

cls_en = classified[classified['language_group'] == 'english'].copy()
cls_ko = classified[classified['language_group'] == 'korean'].copy()

print(f'[{ANALYSIS_METHOD}] 영어: {len(cls_en):,}건, 한국어: {len(cls_ko):,}건')

[anthropic] 영어: 1,649건, 한국어: 298건


In [12]:
colors = {
    'gameplay': '#FF6B6B', 'story_content': '#4ECDC4', 'repetition': '#45B7D1',
    'technical': '#96CEB4', 'company': '#FFEAA7', 'forced_neg': '#DDA0DD', 'other': '#C0C0C0'
}


def category_summary(df, label):
    counts = df['category'].value_counts()
    pcts = df['category'].value_counts(normalize=True) * 100
    summary = pd.DataFrame({'count': counts, 'pct': pcts.round(1)})
    summary = summary.reindex(CATEGORIES).fillna(0)
    summary['count'] = summary['count'].astype(int)
    print(f'\n=== {label} 부정 리뷰 카테고리 ({len(df):,}건) ===')
    for cat in CATEGORIES:
        bar = '█' * int(summary.loc[cat, 'pct'] / 2)
        print(f'  {cat:15s} {summary.loc[cat, "count"]:>5.0f}건 ({summary.loc[cat, "pct"]:>5.1f}%) {bar}')
    return summary


summary_en = category_summary(cls_en, '영어')
summary_ko = category_summary(cls_ko, '한국어')

# ── tone 분포 ──
print(f'\n=== tone 분포 (영어) ===')
print(cls_en['tone'].value_counts().to_string())

# ── actual_sentiment 분포 ──
print(f'\n=== actual_sentiment 분포 (영어) ===')
print(cls_en['actual_sentiment'].value_counts().to_string())


=== 영어 부정 리뷰 카테고리 (1,649건) ===
  gameplay          507건 ( 30.7%) ███████████████
  story_content     165건 ( 10.0%) █████
  repetition        424건 ( 25.7%) ████████████
  technical         169건 ( 10.2%) █████
  company           159건 (  9.6%) ████
  forced_neg         42건 (  2.5%) █
  other             183건 ( 11.1%) █████

=== 한국어 부정 리뷰 카테고리 (298건) ===
  gameplay           78건 ( 26.2%) █████████████
  story_content      39건 ( 13.1%) ██████
  repetition         67건 ( 22.5%) ███████████
  technical          45건 ( 15.1%) ███████
  company            19건 (  6.4%) ███
  forced_neg          7건 (  2.3%) █
  other              43건 ( 14.4%) ███████

=== tone 분포 (영어) ===
tone
constructive    1042
emotional        370
mixed            189
troll             48

=== actual_sentiment 분포 (영어) ===
actual_sentiment
mixed_negative                839
pure_negative                 690
positive_but_negative_vote     80
joke                           40


In [35]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['영어 부정 리뷰', '한국어 부정 리뷰'],
                    shared_yaxes=True)

for i, (summary, label) in enumerate([(summary_en, '영어'), (summary_ko, '한국어')]):
    cats = summary.index.tolist()
    fig.add_trace(
        go.Bar(y=cats, x=summary['pct'], orientation='h',
               marker_color=[colors.get(c, '#999') for c in cats],
               text=[f"{v:.1f}%" for v in summary['pct']],
               textposition='outside', showlegend=False),
        row=1, col=i+1)

fig.update_layout(title=f'부정 리뷰 카테고리 분포 [{ANALYSIS_METHOD}]',
                  height=450, width=900, template='plotly_white')
fig.update_xaxes(title_text='비중 (%)', range=[0, 50])
fig.show()

In [36]:
print(f'=== 카테고리별 대표 리뷰 (영어, {ANALYSIS_METHOD}) ===')
print()

for cat in CATEGORIES:
    subset = cls_en[cls_en['category'] == cat]
    if len(subset) == 0:
        continue
    print(f'--- {cat} ({len(subset)}건) ---')
    good = subset[(subset['text_len'] > 30) & (subset['text_len'] < 500)]
    if 'high' in good['confidence'].values:
        good = good[good['confidence'] == 'high']
    samples = good.head(3) if len(good) >= 3 else subset.head(3)
    for _, row in samples.iterrows():
        text = row['review_text'][:200]
        print(f'  [{row["play_segment"]:8s}] [tone={row["tone"]:12s}] {text}')
        if row['reason'] and row['reason'] != 'parse_failed':
            print(f'           → {row["reason"][:100]}')
    print()

=== 카테고리별 대표 리뷰 (영어, anthropic) ===

--- gameplay (507건) ---
  [engaged(10-30h)] [tone=constructive] Good game but the bosses are near impossible to fight unless you nail the exact strategy.
           → Specific complaint about boss difficulty requiring exact strategies, acknowledges game is good other
  [engaged(10-30h)] [tone=constructive] if you didn't think boss battles and stealth sections would work in a cosy game about swimming around slowly catching fish, you are probably correct
           → Specific criticism of boss battles and stealth sections not fitting the cozy diving gameplay.
  [regular(2-10h)] [tone=constructive] I don't like how too many things are thrown at you immediately at the start of the game. I don't like these kind of management games. It makes me not want to play. I like the ones where you learn slow
           → Specific complaint about overwhelming management mechanics introduced too quickly.

--- story_content (165건) ---
  [engaged(10-30h)] [tone=constru

In [18]:
cls_ko['confidence'].value_counts(dropna=False)

confidence
high      233
medium     54
low        11
Name: count, dtype: int64

In [19]:
import pandas as pd

# 1. Confidence 수치화 (high=3, medium=2, low=1)
conf_mapping = {'high': 3, 'medium': 2, 'low': 1}
cls_ko['conf_score'] = cls_ko['confidence'].str.lower().map(conf_mapping)

# 2. 500자 기준 분리 및 통계 계산
over_500 = cls_ko[cls_ko['text_len'] > 500]['conf_score'].dropna()
under_500 = cls_ko[cls_ko['text_len'] <= 500]['conf_score'].dropna()

print('=== 텍스트 길이별 Confidence 통계 (High=3, Medium=2, Low=1) ===')
if not over_500.empty:
    print(f'500자 초과 ({len(over_500)}건): 평균 {over_500.mean():.2f}, 표준편차 {over_500.std():.2f}')
if not under_500.empty:
    print(f'500자 이하 ({len(under_500)}건): 평균 {under_500.mean():.2f}, 표준편차 {under_500.std():.2f}')
if not over_500.empty and not under_500.empty:
    print(f'평균 차이: {abs(over_500.mean() - under_500.mean()):.2f}')
print('=================================================================\n')

# 3. 카테고리별 리뷰 출력
print(f'=== 카테고리별 대표 리뷰 (한국어, {ANALYSIS_METHOD}) ===\n')

for cat in CATEGORIES:
    subset = cls_ko[cls_ko['category'] == cat]
    if len(subset) == 0:
        continue
    
    print(f'--- {cat} ({len(subset)}건) ---')
    
    good = subset[subset['text_len'] > 500].copy()
    if len(good) == 0:
        print('  (500자 초과 리뷰 없음)\n')
        continue
    
    # 정렬: high -> medium -> low 순서
    good['confidence_cat'] = pd.Categorical(
        good['confidence'].str.lower(), 
        categories=['high', 'medium', 'low'], 
        ordered=True
    )
    good = good.sort_values('confidence_cat')
    
    samples = good.head(3)
    
    for _, row in samples.iterrows():
        conf_val = str(row["confidence"]).upper() if pd.notna(row["confidence"]) else "N/A"
        print(f'  [{row["play_segment"]:8s}] [tone={row["tone"]:12s}] [conf={conf_val:6s}] 길이: {row["text_len"]}자')
        print(f'  {row["review_text"]}')
        if pd.notna(row['reason']) and row['reason'] != 'parse_failed':
            print(f'  → {row["reason"]}')
        print()

=== 텍스트 길이별 Confidence 통계 (High=3, Medium=2, Low=1) ===
500자 초과 (22건): 평균 2.95, 표준편차 0.21
500자 이하 (276건): 평균 2.73, 표준편차 0.53
평균 차이: 0.23

=== 카테고리별 대표 리뷰 (한국어, anthropic) ===

--- gameplay (78건) ---
  [hardcore(50h+)] [tone=constructive] [conf=HIGH  ] 길이: 985자
  잘만든 게임이기는 한데 명성에 비해서는 뒷심이 부족해서 아쉬운 작품.

연출이나 기본 시스템 베이스, 패러디 같은 개그 요소 하나하나가 참 잘만들긴 했으나 중반 이후 편의성에서 게임이 확 무너지는 느낌이 많이 듭니다. 보통 이런 계열에서 뭐가 미비하다라고 하면 플탐 늘리기용 시간박치기 컨텐츠가 대부분일텐데 데이브는 오히려 더 하고싶은데도 이런 답답함이 개발사에서 의도한게 아닐거라는 믿음 속에서 호흡이 끊기는 이유로 제동이 걸리는게 아쉽네요.

어느정도 반복작업이 이루어지는 데이브의 특성상 점점 발전되어가는 스펙에 대해서 만족도가 높은 편인데 중반부터는 극후반에 비해서도 그 체감이 크지않다보니 처음의 답답했던 부분을 끝까지 답답해해야하는게 특히 물고기 습득 시간 소요, 통발 이용 경로 등등 들이는 노력에 비해서 그 수확이 커지지 않다보니 양식장, 농장 자동화를 통해서도 극복하기가 쉽지않습니다. 나이프의 추가 업그레이드가 열렸을 때는 습득 시간을 단축시켜주지 않을까 했는데 광석채취기능같은 부차적인 요소 뿐이었고(이마저도 그 시점엔 크게 활용되는 포인트가 아니었네요.) 통발의 경우 매번 설치하러 가야하고, 매번 기다려야 한다는 점에서 오히려 유저의 동선을 방해하는 컨텐츠가 되어버렸습니다. 설치 후 회수하기까지 절대 시간이 짧게 걸리는게 아닐뿐더러 통발에 사용할 떡밥을 따로 만드는 것 조차도 반쵸스시에 가서 떡밥을 제조후 가져가야하는데 떡밥의 제조부터가 낱개로 만들어야하다보니 왜 일괄갯수설정이 없는지 도통

---
## 9. 플레이타임 구간별 불만 구성

In [37]:
ct_en = pd.crosstab(cls_en['play_segment'], cls_en['category'], normalize='index') * 100
ct_en = ct_en.reindex(index=segment_order, columns=CATEGORIES).fillna(0)

print('=== 플레이타임 구간별 부정 카테고리 (영어) ===')
for seg in segment_order:
    n = len(cls_en[cls_en['play_segment'] == seg])
    print(f'\n{seg} ({n}건):')
    for cat in CATEGORIES:
        pct = ct_en.loc[seg, cat]
        if pct > 0:
            bar = '█' * int(pct / 3)
            print(f'  {cat:15s} {pct:>5.1f}% {bar}')

=== 플레이타임 구간별 부정 카테고리 (영어) ===

casual(<2h) (367건):
  gameplay         27.8% █████████
  story_content     6.8% ██
  repetition       14.2% ████
  technical        20.2% ██████
  company           8.7% ██
  forced_neg        1.9% 
  other            20.4% ██████

regular(2-10h) (390건):
  gameplay         24.6% ████████
  story_content     8.7% ██
  repetition       36.2% ████████████
  technical         9.5% ███
  company           7.9% ██
  forced_neg        2.3% 
  other            10.8% ███

engaged(10-30h) (564건):
  gameplay         34.4% ███████████
  story_content    13.8% ████
  repetition       26.6% ████████
  technical         5.3% █
  company           9.0% ███
  forced_neg        2.8% 
  other             8.0% ██

engaged(30-50h) (188건):
  gameplay         38.3% ████████████
  story_content    10.1% ███
  repetition       25.5% ████████
  technical         8.5% ██
  company           9.0% ███
  forced_neg        2.1% 
  other             6.4% ██

hardcore(50h+) (140건):
  ga

In [38]:
fig = go.Figure()
for cat in CATEGORIES:
    fig.add_trace(go.Bar(
        name=cat, x=segment_order,
        y=[ct_en.loc[seg, cat] for seg in segment_order],
        marker_color=colors.get(cat, '#999'),
        text=[f"{ct_en.loc[seg, cat]:.0f}%" if ct_en.loc[seg, cat] > 5 else '' for seg in segment_order],
        textposition='inside'))

fig.update_layout(barmode='stack', title=f'플레이타임 구간별 부정 카테고리 [{ANALYSIS_METHOD}]',
                  xaxis_title='플레이타임 구간', yaxis_title='비중 (%)',
                  height=500, width=800, template='plotly_white',
                  legend=dict(orientation='h', yanchor='bottom', y=1.02))
fig.show()

In [39]:
# ── 메인스토리 종료 분기 (10-50h를 세분화) ──
engaged_en = cls_en[cls_en['play_segment'].str.startswith('engaged')].copy()

def fine_segment(hours):
    if hours < 20:
        return '10-20h (메인스토리 중)'
    elif hours < 30:
        return '20-30h (종료 전후)'
    else:
        return '30-50h (포스트게임)'

engaged_en['fine_segment'] = engaged_en['playtime_hours'].apply(fine_segment)

print('=== engaged 구간 세분화 (영어) ===')
fine_ct = pd.crosstab(engaged_en['fine_segment'], engaged_en['category'], normalize='index') * 100
fine_ct = fine_ct.reindex(columns=CATEGORIES).fillna(0)

fine_order = ['10-20h (메인스토리 중)', '20-30h (종료 전후)', '30-50h (포스트게임)']
for seg in fine_order:
    if seg not in fine_ct.index:
        continue
    n = len(engaged_en[engaged_en['fine_segment'] == seg])
    print(f'\n{seg} ({n}건):')
    for cat in CATEGORIES:
        pct = fine_ct.loc[seg, cat]
        if pct > 0:
            bar = '█' * int(pct / 3)
            print(f'  {cat:15s} {pct:>5.1f}% {bar}')

=== engaged 구간 세분화 (영어) ===

10-20h (메인스토리 중) (336건):
  gameplay         33.0% ███████████
  story_content    16.1% █████
  repetition       26.2% ████████
  technical         6.0% █
  company           8.6% ██
  forced_neg        2.7% 
  other             7.4% ██

20-30h (종료 전후) (227건):
  gameplay         36.6% ████████████
  story_content    10.6% ███
  repetition       26.9% ████████
  technical         4.4% █
  company           9.7% ███
  forced_neg        3.1% █
  other             8.8% ██

30-50h (포스트게임) (189건):
  gameplay         38.1% ████████████
  story_content    10.1% ███
  repetition       25.9% ████████
  technical         8.5% ██
  company           9.0% ██
  forced_neg        2.1% 
  other             6.3% ██


In [40]:
hardcore_neg = cls_en[cls_en['play_segment'] == 'hardcore(50h+)']

print(f'=== 50시간+ 부정 리뷰 ({len(hardcore_neg)}건) ===')
if len(hardcore_neg) > 0:
    print(f'평균 플레이타임: {hardcore_neg["playtime_hours"].mean():.0f}시간')
    print(f'\n카테고리 분포:')
    print(hardcore_neg['category'].value_counts().to_string())
    print(f'\ntone 분포:')
    print(hardcore_neg['tone'].value_counts().to_string())
    print(f'\n--- 대표 리뷰 ---')
    for _, row in hardcore_neg.head(5).iterrows():
        print(f'[{row["playtime_hours"]:.0f}h] [{row["category"]}] [{row["tone"]}] {row["review_text"][:150]}')
        print()

=== 50시간+ 부정 리뷰 (140건) ===
평균 플레이타임: 85시간

카테고리 분포:
category
gameplay         43
repetition       33
company          28
technical        12
other             9
story_content     9
forced_neg        6

tone 분포:
tone
constructive    81
emotional       28
mixed           26
troll            5

--- 대표 리뷰 ---
[323h] [technical] [constructive] To be clear, I absolutely adore this game, as is evident given the hours I've sunk into it.  Unfortunately I am now locked out from being able to play

[70h] [repetition] [constructive] The Positives: The start of the game is really solid and has a really tight, satisfying gameplay loop where you dive down, collect fish, sell your fis

[52h] [gameplay] [constructive] It was supposed to be a fun game, but honestly the end-game upgrades aren't helpful enough. At some point, I should be able to get faster or have... s

[52h] [repetition] [constructive] Cross-over Dredge/ Subnautica. I kept playing to see if it would be a good game for my kids, fun for a 

---
## 10. 영어 vs 한국어 비교

In [41]:
compare = pd.DataFrame({
    '영어_건수': summary_en['count'],
    '영어_%': summary_en['pct'],
    '한국어_건수': summary_ko['count'],
    '한국어_%': summary_ko['pct'],
}).fillna(0)
compare['차이(pp)'] = (compare['한국어_%'] - compare['영어_%']).round(1)

print('=== 영어 vs 한국어 비교 ===')
print(compare.to_string())

print(f'\n--- 3pp 이상 차이 ---')
for cat in CATEGORIES:
    diff = compare.loc[cat, '차이(pp)']
    if abs(diff) >= 3:
        direction = '한국어↑' if diff > 0 else '영어↑'
        print(f'  {cat}: {direction} ({diff:+.1f}pp)')

=== 영어 vs 한국어 비교 ===
               영어_건수  영어_%  한국어_건수  한국어_%  차이(pp)
category                                         
gameplay         507  30.7      78   26.2    -4.5
story_content    165  10.0      39   13.1     3.1
repetition       424  25.7      67   22.5    -3.2
technical        169  10.2      45   15.1     4.9
company          159   9.6      19    6.4    -3.2
forced_neg        42   2.5       7    2.3    -0.2
other            183  11.1      43   14.4     3.3

--- 3pp 이상 차이 ---
  gameplay: 영어↑ (-4.5pp)
  story_content: 한국어↑ (+3.1pp)
  repetition: 영어↑ (-3.2pp)
  technical: 한국어↑ (+4.9pp)
  company: 영어↑ (-3.2pp)
  other: 한국어↑ (+3.3pp)


In [42]:
cats_plot = [c for c in CATEGORIES if c != 'other']

fig = go.Figure()
fig.add_trace(go.Scatterpolar(
    r=[summary_en.loc[c, 'pct'] if c in summary_en.index else 0 for c in cats_plot],
    theta=cats_plot, fill='toself', name=f'영어 (n={len(cls_en):,})', opacity=0.6))
fig.add_trace(go.Scatterpolar(
    r=[summary_ko.loc[c, 'pct'] if c in summary_ko.index else 0 for c in cats_plot],
    theta=cats_plot, fill='toself', name=f'한국어 (n={len(cls_ko):,})', opacity=0.6))

fig.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0, 50])),
                  title=f'부정 리뷰 프로필 [{ANALYSIS_METHOD}]',
                  height=500, width=600, template='plotly_white')
fig.show()

---

In [44]:
conn.close()
print('DB 연결 종료')

DB 연결 종료


---
##  결과 요약

### 생성된 DB 테이블
| 테이블 | 내용 | 복합키 |
|--------|------|--------|
| `neg_reviews_classified` | 영어+한국어 부정 리뷰 × 최대 3 methods | (review_id, classify_method) |

### 테이블 구조
```
neg_reviews_classified
├── 식별: review_id (PK1), classify_method (PK2), language_group
├── 분류: category, sub_category, tone, actual_sentiment, confidence, reason
├── 유저: play_segment, playtime_hours, num_games_owned, num_reviews
├── 구매: steam_purchase, received_for_free
├── 영향력: votes_up, votes_funny, weighted_vote_score
├── 개발자: has_dev_response
├── 시기: review_month, written_during_early_access
└── 텍스트: review_text, cleaned_text
```

### SQL 쿼리 예시
```sql
-- 건설적 비판만 (기획자용)
SELECT * FROM neg_reviews_classified
WHERE tone = 'constructive' AND classify_method = 'gemini';

-- 긍정인데 비추천 유저
SELECT * FROM neg_reviews_classified
WHERE actual_sentiment = 'positive_but_negative_vote';

-- 20시간 전후 감성 분기
SELECT
    CASE WHEN playtime_hours < 20 THEN '0-20h'
         WHEN playtime_hours < 30 THEN '20-30h'
         ELSE '30h+' END as band,
    category, COUNT(*) as n
FROM neg_reviews_classified
WHERE classify_method = 'gemini'
GROUP BY band, category ORDER BY band, n DESC;

-- 개발자가 어떤 불만에 응답했는가?
SELECT category,
       SUM(has_dev_response) as with_resp,
       COUNT(*) as total,
       ROUND(SUM(has_dev_response)*100.0/COUNT(*),1) as resp_rate
FROM neg_reviews_classified
WHERE classify_method = 'gemini'
GROUP BY category ORDER BY resp_rate DESC;

-- 영향력 큰데 개발자 응답 없는 리뷰
SELECT category, votes_up, weighted_vote_score,
       SUBSTR(review_text,1,100) as preview
FROM neg_reviews_classified
WHERE has_dev_response = 0 AND weighted_vote_score > 0.5
  AND classify_method = 'gemini'
ORDER BY weighted_vote_score DESC LIMIT 20;

-- 3모델 비교: 같은 리뷰의 카테고리 차이
SELECT r.review_id,
       r.category as rules_cat,
       g.category as gemini_cat,
       a.category as anthropic_cat
FROM neg_reviews_classified r
JOIN neg_reviews_classified g ON r.review_id = g.review_id
JOIN neg_reviews_classified a ON r.review_id = a.review_id
WHERE r.classify_method = 'rules'
  AND g.classify_method = 'gemini'
  AND a.classify_method = 'anthropic'
  AND NOT (r.category = g.category AND g.category = a.category)
LIMIT 20;
```